[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/debatelab/practical-deliberation-llms/blob/dev/notebooks/re_sampling_stability_colab.ipynb)

# Re-sampling Stability Experiments (Colab Draft)

This notebook (draft) runs the `experiments/re_sampling_stability` pipeline using a local vLLM OpenAI-compatible server.

In [ ]:
# Execute once per Colab runtime to set up environment and install dependencies.
!uv pip install -q vllm --torch-backend=auto
!uv pip install -q nest_asyncio huggingface_hub # "git+https://github.com/debatelab/practical-deliberation-llms.git@dev"
!git clone --branch dev "https://github.com/debatelab/practical-deliberation-llms.git"
%cd practical-deliberation-llms
!uv pip install -q -e .
!uv pip list | grep "vllm\|nest-asyncio\|huggingface-hub\|practical-deliberation-llms"

OSError: [Errno 5] Input/output error

In [ ]:
#@title Environment, configuration, and Start button (run this cell once, then tweak and click Start)
#@markdown This cell installs dependencies (once per runtime), configures the experiment via form fields,
#@markdown and defines a Start button that will clone the repo (if needed), start vLLM, run the experiment,
#@markdown and inspect results using the current settings.

import os, json, asyncio, time, subprocess, requests
import nest_asyncio
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display

# ---------------------------------------------------------------------------
# Configuration (Colab form fields)
# ---------------------------------------------------------------------------
# --- vLLM / model settings ---
model_name = "Qwen/Qwen3-1.7B"  #@param {type:"string"}
vllm_port = 8000  #@param {type:"integer"}
# Optional: if set, treat model_name as a LoRA adapter on top of this base.
base_model_name = ""  #@param {type:"string"}

# --- ExperimentConfig core settings (thin mapping) ---
dataset_name = "daily_dilemmas"  #@param ["daily_dilemmas","AITA","RoleConflictBench","AIRiskDilemmas","MSMDilemmas","legalbench_corporate_lobbying","legalbench_insurance_policy_interpretation"]
n_problems = 10  #@param {type:"integer"}
adapter_kwargs_json = "{}"  #@param ["{}","{'framework':'default'}","{}","{'name':'default','split':'train'}","{'subset':'model_eval','split':'test'}","{}","{}"] {allow-input: true}

# Generation and sampling.
n_traces_per_problem = 8 #@param {type:"integer"}
seed = 1234  #@param {type:"integer"}
temperature = 0.7  #@param {type:"number"}
top_p = 1.0  #@param {type:"number"}
max_concurrency = 8  #@param {type:"integer"}

# Transformations.
max_transformations_per_problem = 0  #@param {type:"integer"}
use_action_permutation_transform = False  #@param {type:"boolean"}

# Models used inside ExperimentConfig. Use the same underlying model for both roles.
candidate_model = model_name
assistant_model = model_name

# Output directory and misc.
output_dir_suffix = "colab_run"  #@param {type:"string"}
make_plots = True  #@param {type:"boolean"}
results_file_format = "jsonl"  #@param ["jsonl", "parquet"]
log_level = "INFO"  #@param ["DEBUG", "INFO", "WARNING", "ERROR"]

# Optional: override base URL / API token (usually left blank when using vLLM).
openai_base_url_override = ""  #@param {type:"string"}
api_token_override = ""  #@param {type:"string"}

# ---------------------------------------------------------------------------
# Helper functions used by the Start button
# ---------------------------------------------------------------------------
from experiments.re_sampling_stability.config import ExperimentConfig, DatasetSpec
from experiments.re_sampling_stability.run_experiment import run_experiment_async, resolve_callable

def _infer_base_model_from_lora(repo_id: str) -> str | None:
    """Best-effort heuristic: try to infer base model from a LoRA repo.

    If we cannot infer confidently, return None and rely on the
    user-provided base_model_name.
    """
    try:
        from huggingface_hub import hf_hub_download
    except Exception:
        return None

    candidate_files = ["adapter_config.json", "adapter_config.bin"]
    for fname in candidate_files:
        try:
            local_path = hf_hub_download(repo_id, fname)
        except Exception:
            continue
        try:
            with open(local_path, "r", encoding="utf-8") as f:
                cfg = json.load(f)
        except Exception:
            continue
        for key in ("base_model_name_or_path", "base_model_name", "base_model"):
            val = cfg.get(key)
            if isinstance(val, str) and val.strip():
                return val.strip()
    return None

def start_vllm_server() -> tuple[subprocess.Popen, str]:
    """Start a vLLM OpenAI-compatible server (plain or LoRA) and return (process, base_url).

    This uses the current form variables model_name, base_model_name, and vllm_port.
    """
    vllm_base_url = f"http://127.0.0.1:{vllm_port}/v1"

    effective_base_model_name = base_model_name.strip() or None
    if effective_base_model_name is None:
        inferred = _infer_base_model_from_lora(model_name)
        if inferred is not None:
            print(f'Inferred base_model_name={inferred!r} from LoRA repo {model_name!r}.')
            effective_base_model_name = inferred

    if effective_base_model_name:
        lora_descriptor = {
            "name": model_name,
            "path": model_name,
            "base_model_name": effective_base_model_name,
        }
        cmd = [
            "vllm", "serve", effective_base_model_name,
            "--trust-remote-code",
            "--dtype", "half",
            "--max-model-len", "16384",
            "--enable-chunked-prefill",
            "--tensor-parallel-size", "1",
            "--port", str(vllm_port),
            "--enable-lora",
            "--lora-modules", json.dumps(lora_descriptor),
        ]
        print("Starting vLLM in LoRA mode with base_model=", effective_base_model_name, "and adapter=", model_name)
    else:
        cmd = [
            "vllm", "serve", model_name,
            "--trust-remote-code",
            "--dtype", "half",
            "--max-model-len", "16384",
            "--enable-chunked-prefill",
            "--tensor-parallel-size", "1",
            "--port", str(vllm_port),
        ]
        print("Starting vLLM in plain model mode with model=", model_name)

    vllm_log_path = os.path.join(os.getcwd(), 'vllm_server.log')
    vllm_log_file = open(vllm_log_path, 'w', buffering=1, encoding='utf-8')
    proc = subprocess.Popen(
        cmd,
        stdout=vllm_log_file,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(f'vLLM server logs will be written to: {vllm_log_path}')

    for _ in range(60):
        try:
            r = requests.get(vllm_base_url + "/models", timeout=1)
            if r.ok:
                data = r.json().get("data", [])
                model_ids = [m.get("id") for m in data]
                print("vLLM server is ready. Available models:", model_ids)
                break
        except Exception:
            time.sleep(10)
    else:
        print("Warning: vLLM server did not become ready in time.")

    os.environ.setdefault('OPENAI_BASE_URL', vllm_base_url)
    os.environ.setdefault('OPENAI_API_KEY', 'dummy-key')
    return proc, vllm_base_url

def build_experiment_config() -> ExperimentConfig:
    """Build an ExperimentConfig from the current form variables.
    """
    try:
        adapter_kwargs = json.loads(adapter_kwargs_json) if adapter_kwargs_json.strip() else {}
    except Exception as e:
        raise ValueError(f'Failed to parse adapter_kwargs_json: {e}')

    datasets = [
        DatasetSpec(
            name=dataset_name,
            n_problems=n_problems,
            adapter_kwargs=adapter_kwargs,
        )
    ]

    if use_action_permutation_transform:
        transform_problems_fn_path = 'experiments.re_sampling_stability.transform.permutate_actions'
    else:
        transform_problems_fn_path = None

    base_output_dir = 'experiments/re_sampling_stability/outputs'
    output_dir = os.path.join(base_output_dir, output_dir_suffix)

    config = ExperimentConfig(
        candidate_model=candidate_model,
        assistant_model=assistant_model,
        openai_base_url=openai_base_url_override or None,
        api_token=api_token_override or None,
        seed=seed,
        datasets=datasets,
        n_traces_per_problem=n_traces_per_problem,
        temperature=temperature,
        top_p=top_p,
        max_concurrency=max_concurrency,
        max_transformations_per_problem=max_transformations_per_problem,
        output_dir=output_dir,
        make_plots=make_plots,
        results_file_format=results_file_format,
        log_level=log_level,
        transform_problems_fn=transform_problems_fn_path,
        generate_reasoning_trace_fn='experiments.re_sampling_stability.reasoning.generate_reasoning_traces_for_problem',
        score_choice_labels_fn='experiments.re_sampling_stability.judgment.score_choice_labels_for_trace',
    )
    return config

class NotebookProgress:
    """Simple progress reporter for Colab.

    Prints the current step whenever the stage changes and shows tqdm
    notebook progress bars for reasoning (per problem) and scoring
    (per trace).
    """

    def __init__(self):
        self._current_stage = None
        self._reasoning_bar = None
        self._scoring_bar = None

    def __call__(self, stage: str, completed: int, total: int) -> None:
        if stage != self._current_stage:
            self._current_stage = stage
            friendly = {
                'load_problems': 'Loading problems',
                'transform_problems': 'Transforming problems',
                'reasoning': 'Generating reasoning traces',
                'scoring': 'Scoring labels',
                'finalize': 'Computing metrics & saving results',
            }.get(stage, stage)
            print(f'Current step: {friendly}')

        if stage == 'reasoning':
            if total <= 0:
                return
            if self._reasoning_bar is None:
                self._reasoning_bar = tqdm(
                    total=total,
                    desc='Problems (reasoning)',
                    position=0,
                    leave=True,
                )
            delta = completed - self._reasoning_bar.n
            if delta > 0:
                self._reasoning_bar.update(delta)
            return

        if stage == 'scoring':
            if total <= 0:
                return
            if self._scoring_bar is None:
                self._scoring_bar = tqdm(
                    total=total,
                    desc='Traces (scoring)',
                    position=1,
                    leave=True,
                )
            delta = completed - self._scoring_bar.n
            if delta > 0:
                self._scoring_bar.update(delta)

def run_experiment_with_progress(config: ExperimentConfig) -> None:
    """Run the experiment with a notebook progress bar and basic logging.
    """
    try:
        from practical_deliberation_llms.inference import InferenceError
    except ImportError:
        InferenceError = Exception

    transform_fn = None
    if config.transform_problems_fn:
        transform_fn = resolve_callable(config.transform_problems_fn)

    generate_reasoning_fn = resolve_callable(config.generate_reasoning_trace_fn)
    score_labels_fn = resolve_callable(config.score_choice_labels_fn)

    first_ds = config.datasets[0]
    progress = NotebookProgress()
    print('Starting experiment...')
    print(f'  Dataset: {first_ds.name} (n_problems={first_ds.n_problems})')
    print(f'  Model: {config.candidate_model}')
    print(f'  Output dir: {config.output_dir}')

    async def _run():
        await run_experiment_async(
            config=config,
            transform_problems_fn=transform_fn,
            generate_reasoning_trace_fn=generate_reasoning_fn,
            score_choice_labels_fn=score_labels_fn,
            progress_cb=progress,
        )

    nest_asyncio.apply()
    t0 = time.time()
    try:
        asyncio.run(_run())
    except InferenceError as e:
        print('Experiment failed due to an inference error:')
        print(f'  {e}')
        raise
    except Exception as e:
        print('Experiment failed with an unexpected error:')
        print(f'  {e}')
        raise
    else:
        dt = time.time() - t0
        print(f'Experiment finished successfully in {dt:.1f} seconds.')
        print(f'Results written to: {config.output_dir}')

def inspect_results(config: ExperimentConfig) -> None:
    """Inspect results for a finished experiment (metrics, plots, samples).
    """
    import pandas as pd
    from IPython.display import Image, display as ipy_display

    out_dir = config.output_dir
    print('Inspecting results in:', out_dir)

    def _load_df(base_name: str):
        ext = config.results_file_format
        path = os.path.join(out_dir, f'{base_name}.{ext}')
        if not os.path.exists(path):
            print(f'Missing {base_name} file at {path}')
            return None
        if ext == 'jsonl':
            return pd.read_json(path, lines=True)
        if ext == 'parquet':
            return pd.read_parquet(path)
        print(f'Unsupported results_file_format={ext!r}')
        return None

    metrics_within = _load_df('metrics_within')
    if metrics_within is not None and 'jsd_information_radius' in metrics_within.columns:
        print('Within-context JSD information radius summary (per problem):')
        ipy_display(metrics_within['jsd_information_radius'].describe(percentiles=[0.25, 0.5, 0.75]))
        if 'source_dataset' in metrics_within.columns:
            print('Per-dataset JSD summary:')
            ipy_display(metrics_within.groupby('source_dataset')['jsd_information_radius'].describe())
    else:
        print('No metrics_within data or jsd_information_radius column not found.')

    metrics_bvt = _load_df('metrics_baseline_vs_trans')
    if metrics_bvt is not None and 'D_KL_base_to_trans' in metrics_bvt.columns:
        print('Baseline vs transformed KL divergence summary:')
        ipy_display(metrics_bvt['D_KL_base_to_trans'].describe(percentiles=[0.25, 0.5, 0.75]))
    else:
        print('No baseline_vs_trans metrics or D_KL_base_to_trans column not found.')

    jsd_plot_path = os.path.join(out_dir, 'jsd_information_radius_boxplot.png')
    if os.path.exists(jsd_plot_path):
        print('Within-context JSD information radius boxplot:')
        ipy_display(Image(jsd_plot_path))
    else:
        print('JSD boxplot not found (make_plots may be False or metrics missing).')

    traces = _load_df('traces')
    if traces is not None:
        print('Sample of traces:')
        ipy_display(traces.head())
    scores = _load_df('scores')
    if scores is not None:
        print('Sample of scores:')
        ipy_display(scores.head())

# ---------------------------------------------------------------------------
# Start button wiring
# ---------------------------------------------------------------------------
start_experiment_button = widgets.Button(
    description='Start experiment',
    button_style='primary',
    tooltip='Start vLLM and run the experiment with current settings',
)

_is_running = False

def _on_start_experiment_clicked(_):
    global _is_running, vllm_process
    if _is_running:
        print('Experiment already running; please wait for it to finish.')
        return
    _is_running = True
    try:
        print('=== Starting experiment (orchestrated) ===')

        if 'vllm_process' in globals() and vllm_process is not None and vllm_process.poll() is None:
            print('vLLM server already running; reusing existing server.')
        else:
            vllm_process, _ = start_vllm_server()

        config = build_experiment_config()
        print('ExperimentConfig summary:')
        first_ds = config.datasets[0]
        print(f'  Dataset: {first_ds.name} (n_problems={first_ds.n_problems})')
        print(f'  Model: {config.candidate_model}')
        print(f'  Assistant model: {config.assistant_model}')
        print(f'  Output dir: {config.output_dir}')

        run_experiment_with_progress(config)
        inspect_results(config)
    finally:
        _is_running = False

start_experiment_button.on_click(_on_start_experiment_clicked)
display(start_experiment_button)
